# Detecting Crop Disease from Leaf Images


## Importing the Library


In [33]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

## Connecting to Google Drive

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data Augmentation (Preparing the data)

In [35]:
augmentation = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    zoom_range = 0.2,
    horizontal_flip = True,
    validation_split = 0.2
)

In [36]:
train_set = augmentation.flow_from_directory(
    "/content/drive/MyDrive/NBICT LAB PDSML-B-8/8. Plant Disease Detection/Dataset",
    target_size = (224, 224),
    batch_size = 32,
    class_mode = "categorical",
    subset = "training"
    )

Found 1738 images belonging to 3 classes.


In [37]:
val_set = augmentation.flow_from_directory(
    "/content/drive/MyDrive/NBICT LAB PDSML-B-8/8. Plant Disease Detection/Dataset",
    target_size = (224, 224),
    batch_size = 32,
    class_mode = "categorical",
    subset = "validation"
    )

Found 433 images belonging to 3 classes.


## Building the model

In [38]:
from tensorflow.keras import models, layers

**1st layer**
- filters/kernel -> 3*3 pixel filter and this is the kernel size.
- we will create 32 filter hare. filter works differently. but the basic work is to scan.
* 1st filter -> where is vein
* 2nd filter -> how much age
* 3rd filter -> is there any prominent color (brown)
and so on..

activation relu:
* if 1st filter get the negetive value. means he is unable to find patteren. then the information will not pass to the 2nd stage.

input shape:
* all image convert into 150 to 150 pixel and RGB(3) color format. so that we can scan better.

Maxpulling:
* after the filter we create the layer. later in maxpulling we again filter with 2*2 filter. By this avoid the unnecessery information.

* As the image is more or less same thats why we create many layers. if we just want to differentiate just the cat and dog. 2 layer will be enough.

!!! find out why more is bad....

In [39]:
model = models.Sequential([
    # Layer-1: Detects basic edges and the veins of the leaf
    layers.Conv2D(filters=32, kernel_size = (3, 3), activation = 'relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(pool_size = (2, 2)),

    # Layer 2: Detects color blobs and spot pattern
    layers.Conv2D(filters=64, kernel_size = (3, 3), activation = 'relu'),
    layers.MaxPooling2D(pool_size = (2, 2)),

    # Layer 3: Complex shapes (like the halo around a fungus spot)
    layers.Conv2D(filters=128, kernel_size = (3, 3), activation = 'relu'),
    layers.MaxPooling2D(pool_size = (2, 2)),

    # Flatten the 2D image into a 1D line to take the final decision
    layers.Flatten(),
    layers.Dense(units = 128, activation = 'relu'),
    layers.Dropout(0.5),
    layers.Dense(units = 3, activation = 'softmax')

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [40]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_25 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_25 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_26 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,347 (42.61 MB)

 Trainable params: 11,169,347 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## Training the model

In [44]:
history = model.fit(train_set, epochs=10, validation_data=val_set)

Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 381s 7s/step - accuracy: 0.6692 - loss: 0.8319 - val_accuracy: 0.8222 - val_loss: 0.4252
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 272s 5s/step - accuracy: 0.8533 - loss: 0.3865 - val_accuracy: 0.8522 - val_loss: 0.4018
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 255s 5s/step - accuracy: 0.8855 - loss: 0.3055 - val_accuracy: 0.8545 - val_loss: 0.3875
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 253s 5s/step - accuracy: 0.9160 - loss: 0.2379 - val_accuracy: 0.9330 - val_loss: 0.2243
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 275s 5s/step - accuracy: 0.9298 - loss: 0.2008 - val_accuracy: 0.9192 - val_loss: 0.2038
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 257s 5s/step - accuracy: 0.9120 - loss: 0.2310 - val_accuracy: 0.9446 - val_loss: 0.2069
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 268s 5s/step - accuracy: 0.9425 - loss: 0.1779 - val_accuracy: 0.9007 - val_loss: 0.3601
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 251s 5s/step - accuracy: 0.9292 - loss: 0.1987 - val_accuracy: 0.9469 - v